# Agentic AI Meeting Preparation Assistant

## Objective

Build an AI assistant that helps users prepare for upcoming client meetings.

The assistant demonstrates:

- Retrieval Augmented Generation (RAG)
- Vector Database (FAISS)
- Agentic Workflow
- Tool Usage
- Short-Term Memory
- Long-Term Memory

## Example Query

Prepare me for my meeting with Acme Corp.

## Output

- Company Overview
- Previous Meetings
- Open Action Items
- Stakeholders
- Risks
- Opportunities
- Talking Points
- Next Steps

IMPORTS

In [2]:
import os
import json
import faiss
import numpy as np
import pandas as pd

from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer

from langchain_core.documents import Document
from langchain_core.chat_history import InMemoryChatMessageHistory

from langchain_groq import ChatGroq

/home/nineleaps/Documents/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LOADING ENVIRONMENT VARIABLES

In [3]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("Groq API Key not found")

print("Groq API Loaded Successfully")

Groq API Loaded Successfully


### CREATE DATAFRAMES

COMPANY DATA

In [4]:
companies_df = pd.DataFrame({

    "company_name":[
        "Acme Corp",
        "Globex",
        "Initech",
        "Umbrella Health",
        "Wayne Enterprises"
    ],

    "industry":[
        "Manufacturing",
        "Retail",
        "Software",
        "Healthcare",
        "Industrial"
    ],

    "strategic_focus":[
        "AI Logistics",
        "Customer Analytics",
        "Cloud Migration",
        "Patient Analytics",
        "Predictive Maintenance"
    ],

    "recent_news":[
        "Expanding into Asia",
        "Launching loyalty platform",
        "Cloud modernization initiative",
        "Patient analytics platform",
        "Smart factory investments"
    ]
})

MEETING NOTES

In [5]:
meeting_notes_df = pd.DataFrame({

    "client":[
        "Acme Corp",
        "Acme Corp",
        "Globex"
    ],

    "date":[
        "2025-02-10",
        "2025-04-12",
        "2025-05-01"
    ],

    "notes":[
        "Discussed AI logistics implementation.",
        "Requested pricing proposal.",
        "Reviewed analytics roadmap."
    ]
})

ACTION ITEMS

In [6]:
action_items_df = pd.DataFrame({

    "client":[
        "Acme Corp",
        "Acme Corp",
        "Globex"
    ],

    "action":[
        "Send Proposal",
        "Schedule Workshop",
        "Share Case Study"
    ],

    "status":[
        "Open",
        "Open",
        "Closed"
    ]
})

CONTACTS

In [7]:
contacts_df = pd.DataFrame({

    "client":[
        "Acme Corp",
        "Acme Corp"
    ],

    "name":[
        "Sarah Johnson",
        "Michael Chen"
    ],

    "role":[
        "Procurement Director",
        "CTO"
    ]
})

SHORT TERM MEMORY

In [8]:
chat_history = InMemoryChatMessageHistory()

LONG TERM MEMORY

In [9]:
MEMORY_FILE = "memory.json"

#Create Memory File

def save_memory(data):

    with open(MEMORY_FILE, "w") as f:
        json.dump(data, f, indent=4)

#Load Memory

def load_memory():

    with open(MEMORY_FILE, "r") as f:
        return json.load(f)

#Initialize

if not os.path.exists(MEMORY_FILE):

    default_memory = {

        "preferred_style":"concise",

        "important_clients":[],

        "meeting_history":[],

        "user_preferences":[]
    }

    save_memory(default_memory)

CREATING KNOWLEDGE BASED DOCUMENTS

In [35]:
documents = []

for _, row in companies_df.iterrows():

    text = f"""
    Company Name: {row['company_name']}
    Industry: {row['industry']}
    Strategic Focus: {row['strategic_focus']}
    Recent News: {row['recent_news']}
    """

    documents.append(
        Document(page_content=text)
    )

#CONVERTING TO TEXTS
texts = [
    doc.page_content
    for doc in documents
]

print(texts)


['\n    Company Name: Acme Corp\n    Industry: Manufacturing\n    Strategic Focus: AI Logistics\n    Recent News: Expanding into Asia\n    ', '\n    Company Name: Globex\n    Industry: Retail\n    Strategic Focus: Customer Analytics\n    Recent News: Launching loyalty platform\n    ', '\n    Company Name: Initech\n    Industry: Software\n    Strategic Focus: Cloud Migration\n    Recent News: Cloud modernization initiative\n    ', '\n    Company Name: Umbrella Health\n    Industry: Healthcare\n    Strategic Focus: Patient Analytics\n    Recent News: Patient analytics platform\n    ', '\n    Company Name: Wayne Enterprises\n    Industry: Industrial\n    Strategic Focus: Predictive Maintenance\n    Recent News: Smart factory investments\n    ']


GENERATING EMBEDDINGS

In [36]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    texts
)

print(embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4996.34it/s]


(5, 384)


BUILDING FAISS INDEX

In [37]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(
        embeddings,
        dtype=np.float32
    )
)
print("Total Number of Index created:", index.ntotal)

Total Number of Index created: 5


### TOOL DEFINITIONS

COMPANY SEARCH

In [14]:
def company_search(company_name):

    query_embedding = embedding_model.encode(
        [company_name]
    )

    distances, indices = index.search(
        np.array(
            query_embedding,
            dtype=np.float32
        ),
        3
    )

    results = []

    for idx in indices[0]:
        results.append(texts[idx])

    return "\n".join(results)

MEETING NOTES

In [15]:
def meeting_notes(company_name):

    result = meeting_notes_df[
        meeting_notes_df["client"]
        ==
        company_name
    ]

    return result.to_string(index=False)

ACTION ITEMS

In [34]:
def action_items(company_name):

    result = action_items_df[
        action_items_df["client"]
        ==
        company_name
    ]

    return result.to_string(index=False)

CONTACTS

In [17]:
def contacts(company_name):

    result = contacts_df[
        contacts_df["client"]
        ==
        company_name
    ]

    return result.to_string(index=False)

MEMORY TOOL

In [18]:
def memory_tool():

    return json.dumps(
        load_memory(),
        indent=2
    )

### TOOL SCHEMAS

In [19]:
AVAILABLE_TOOLS = [

    "company_search",

    "meeting_notes",

    "action_items",

    "contacts",

    "memory_tool"
]

### PLANNER AGENT

INITIALIZING GROQ

In [20]:
llm = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model="llama-3.3-70b-versatile",
    temperature=0.1
)

PLANNER PROMPT

In [21]:
PLANNER_PROMPT = """
You are a planning agent.

Available tools:

1. company_search
2. meeting_notes
3. action_items
4. contacts
5. memory_tool

Return JSON only.

Example:

{
    "tools":[
        {
            "tool_name":"company_search",
            "company_name":"Acme Corp"
        }
    ]
}
"""

PLANNER FUNCTION

In [30]:
def create_plan(user_query):

    prompt = f"""
    {PLANNER_PROMPT}

    User Query:
    {user_query}
    """

    response = llm.invoke(prompt)

    print("Planner Response:")
    print(response.content)

    return response.content

In [31]:
import json
import re

def parse_plan(plan_text):

    try:

        # Remove markdown fences
        plan_text = plan_text.replace(
            "```json",
            ""
        )

        plan_text = plan_text.replace(
            "```",
            ""
        )

        plan_text = plan_text.strip()

        # Extract JSON object
        match = re.search(
            r"\{.*\}",
            plan_text,
            re.DOTALL
        )

        if match:
            plan_text = match.group()

        return json.loads(plan_text)

    except Exception as e:

        print("Plan Parsing Error:")
        print(e)

        print("\nRaw Response:")
        print(plan_text)

        return {
            "tools":[]
        }

### TOOL EXECUTOR

In [23]:
def execute_tool(tool_name, company_name=None):

    if tool_name == "company_search":
        return company_search(company_name)

    elif tool_name == "meeting_notes":
        return meeting_notes(company_name)

    elif tool_name == "action_items":
        return action_items(company_name)

    elif tool_name == "contacts":
        return contacts(company_name)

    elif tool_name == "memory_tool":
        return memory_tool()

    return "Unknown Tool"

EXECUTE PLAN

In [32]:
def execute_plan(plan):

    parsed = parse_plan(plan)

    tool_results = {}

    for step in parsed["tools"]:

        tool_name = step["tool_name"]

        company_name = step.get(
            "company_name"
        )

        tool_results[tool_name] = execute_tool(
            tool_name,
            company_name
        )

    return tool_results

### MEETING BRIEF GENERATOR

In [25]:
def generate_meeting_brief(
    user_query,
    tool_results
):

    prompt = f"""
    Create an executive meeting brief.

    User Query:
    {user_query}

    Retrieved Context:

    {json.dumps(tool_results, indent=2)}

    Include:

    1. Company Overview
    2. Previous Meetings
    3. Open Action Items
    4. Stakeholders
    5. Risks
    6. Opportunities
    7. Talking Points
    8. Next Steps
    """

    response = llm.invoke(prompt)

    return response.content

MAIN AGENT WORKFLOW

In [26]:
def meeting_preparation_agent(
    user_query
):

    chat_history.add_user_message(
        user_query
    )

    plan = create_plan(
        user_query
    )

    tool_results = execute_plan(
        plan
    )

    final_response = (
        generate_meeting_brief(
            user_query,
            tool_results
        )
    )

    chat_history.add_ai_message(
        final_response
    )

    return final_response

RUNNING END TO END DEMO

In [33]:
user_query = "Prepare me for my meeting with Acme Corp"

response = meeting_preparation_agent(user_query
)

print(response)

Planner Response:
```
{
    "tools":[
        {
            "tool_name":"company_search",
            "company_name":"Acme Corp"
        },
        {
            "tool_name":"meeting_notes",
            "company_name":"Acme Corp"
        },
        {
            "tool_name":"contacts",
            "company_name":"Acme Corp"
        }
    ]
}
```
**Executive Meeting Brief: Acme Corp**

**1. Company Overview**
Acme Corp is a manufacturing company with a strategic focus on AI Logistics. They have recently expanded into Asia, indicating a growing presence in the global market.

**2. Previous Meetings**
We have had two previous meetings with Acme Corp:
- February 10, 2025: Discussed AI logistics implementation.
- April 12, 2025: Requested a pricing proposal.

**3. Open Action Items**
Based on the previous meetings, the open action item is to provide a pricing proposal for the AI logistics implementation.

**4. Stakeholders**
The identified stakeholders from Acme Corp are:
- Sarah Johnson: P